# Dependencies

In [26]:
import os

In [2]:
import pandas as pd
import numpy as np

## Pre-Processing

In [3]:
from sklearn.model_selection import train_test_split

# Declarations

In [27]:
RAW_DATA_FOLDER_NAME = "parquet"
NO_INF_DATA_FOLDER_NAME = "parquet_no_inf"
SPLIT_DATA_FOLDER_NAME = "parquet-split"
LABEL_COLUMN = "Label"
TRAINING_DATA_RATIO = 0.6
VALIDATION_DATA_RATIO = 0.2
TESTING_DATA_RATIO = 0.2

# Pre-Processing

## Change Infinite Values to NaN


Infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) are converted to `NaN` values to ensure that they can be handled consistently during the subsequent missing-value imputation process. This transformation allows both originally missing values and invalid infinite values to be processed using the same imputation method.

In [5]:
def calculate_inf_values(
    source_folder_name: str
):
    parquet_files = sorted(file for file in os.listdir(source_folder_name) if file.endswith(".parquet"))
    total = len(parquet_files)
    count = 0
    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        df = pd.read_parquet(source_file)
        inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
        count += inf_count
    print("\n")
    print(f"Completed. Processed {total} files.")
    print(f"Found {count} inf values")

In [6]:
calculate_inf_values(RAW_DATA_FOLDER_NAME)

Processing [168/168] Wednesday-28-02-2018_TrafficForML_CICFlowMeter_00007.parquet...

Completed. Processed 168 files.
Found 121886.0 inf values


In [7]:
def change_inf_to_nan(
    source_folder_name: str,
    target_folder_name: str
):
    os.makedirs(target_folder_name, exist_ok=True)

    parquet_files = sorted(file for file in os.listdir(source_folder_name) if file.endswith(".parquet"))

    total = len(parquet_files)

    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        target_file = os.path.join(target_folder_name, file_name)

        df = pd.read_parquet(source_file)

        df.replace([np.inf, -np.inf],np.nan,inplace=True)
        df.to_parquet(target_file,index=False)

    print(f"\nCompleted. Processed {total} files.")

In [8]:
change_inf_to_nan(RAW_DATA_FOLDER_NAME, NO_INF_DATA_FOLDER_NAME)

Processing [168/168] Wednesday-28-02-2018_TrafficForML_CICFlowMeter_00007.parquet...
Completed. Processed 168 files.


In [9]:
calculate_inf_values(NO_INF_DATA_FOLDER_NAME)

Processing [168/168] Wednesday-28-02-2018_TrafficForML_CICFlowMeter_00007.parquet...

Completed. Processed 168 files.
Found 0.0 inf values


The dataset was cleaned by converting both positive infinite  and negative infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) to `NaN`. A total of **121,886 infinite values** were identified across **168 Parquet files** before the cleaning process. After the transformation, no infinite values remained in the dataset.

**Before cleaning:**

```text
Completed. Processed 168 files.
Found 121886.0 inf values
```

**After cleaning:**

```text
Completed. Processed 168 files.
Found 0.0 inf values
```

This ensures that all infinite values are handled as missing values and can subsequently be processed during the missing-value imputation stage.


## Train/Validation/Test Split

The CSE-CIC-IDS2018 dataset requires careful consideration when dividing the data into training, validation, and test sets. This is because the attack classes are not uniformly distributed across the dataset; instead, specific attack scenarios were conducted on particular dates. As a result, directly splitting the dataset based on individual days may cause some attack classes to be absent from one or more subsets.

The distribution of attack scenarios across the data collection dates is presented below:

| Date  | Attack(s)                           |
|-------|-------------------------------------|
| 14-02 | FTP-BruteForce, SSH-Bruteforce      |
| 15-02 | DoS-GoldenEye, DoS-Slowloris        |
| 16-02 | DoS-SlowHTTPTest, DoS-Hulk          |
| 20-02 | DDoS-LOIC-HTTP, DDoS-LOIC-UDP       |
| 21-02 | DDoS-LOIC-UDP, DDoS-HOIC            |
| 22-02 | Web Brute Force, XSS, SQL Injection |
| 23-02 | Web Brute Force, XSS, SQL Injection |
| 28-02 | Infiltration                        |
| 01-03 | Infiltration                        |
| 02-03 | Bot                                 |

This distribution indicates that several attack classes are associated with only one or a small number of collection dates. Therefore, assigning entire dates directly to the training, validation, or test set could result in certain attack classes being completely absent from the training data. Such a split would make the experiment evaluate unseen attack-class generalization rather than the intended robustness of the ML-IDS against input disturbances.

Therefore, the primary experiment uses a **stratified train/validation/test split based on the attack label**, ensuring that the attack classes are represented across the three subsets. The validation and test sets are kept separate from the training data to prevent information leakage during model development and final evaluation.

A separate day- or scenario-based split may subsequently be used as an additional experiment to evaluate the model's ability to generalize to traffic collected under different attack scenarios.


In [10]:
def create_train_dev_test_folder(source_folder_name:str, target_folder_name:str,target_day):
    parquet_files = [
        file
        for file in os.listdir(source_folder_name)
        if file.endswith(".parquet") and target_day in file
    ]

    parquet_files = sorted(parquet_files)

    if not parquet_files:
        print(f"No Parquet files found for {target_day}")
        raise ValueError(f"No Parquet files found for {target_day}")
    print(f"Found {len(parquet_files)} files for {target_day}")

    day_folder = os.path.join(
        target_folder_name,
        f"day-{target_day}"
    )

    train_folder = os.path.join(day_folder, "train")
    dev_folder = os.path.join(day_folder, "dev")
    test_folder = os.path.join(day_folder, "test")

    os.makedirs(train_folder, exist_ok=True)
    os.makedirs(dev_folder, exist_ok=True)
    os.makedirs(test_folder, exist_ok=True)

    return parquet_files, train_folder,dev_folder,test_folder

In [11]:
def split_parquet_files(
    source_folder_name: str,
    parquet_files: list[str],
    train_folder: str,
    dev_folder: str,
    test_folder: str,
    dev_size: float = 0.20,
    test_size: float = 0.20,
    label_column: str = "Label",
    random_state: int = 42
):

    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{len(parquet_files)}] {file_name}...", end="\r",flush=True)
        source_file = os.path.join(source_folder_name,file_name)

        df = pd.read_parquet(source_file)

        train_df, temp_df = train_test_split(
            df,
            test_size=(dev_size+test_size),
            random_state=random_state,
            stratify=df[label_column]
        )

        dev_df, test_df = train_test_split(
            temp_df,
            test_size=dev_size/(dev_size+test_size),
            random_state=random_state,
            stratify=temp_df[label_column]
        )

        train_df.to_parquet(
            os.path.join(train_folder, file_name),
            index=False
        )

        dev_df.to_parquet(
            os.path.join(dev_folder, file_name),
            index=False
        )

        test_df.to_parquet(
            os.path.join(test_folder, file_name),
            index=False
        )

    print("\nCompleted.")

In [12]:
def split_a_single_day(
    source_folder_name: str,
    target_folder_name: str,
    target_day: str,
    train_size: float = 0.70,
    dev_size: float = 0.15,
    test_size: float = 0.15,
    label_column: str = LABEL_COLUMN,
    random_state: int = 42
):
    if train_size < 0.0 or dev_size < 0.0 or test_size < 0.0:
        raise ValueError("train_size, dev_size, test_size must be a positive float")
    if train_size + dev_size + test_size != 1.0:
        raise ValueError("train_size + dev_size + test_size must equal 1.0")

    parquet_files, train_folder,dev_folder,test_folder = (
        create_train_dev_test_folder(source_folder_name, target_folder_name, target_day))

    split_parquet_files(source_folder_name,parquet_files,train_folder,dev_folder,test_folder,dev_size,test_size,label_column,random_state)

| Date  | Attack(s)                           |
|-------|-------------------------------------|
| 14-02 | FTP-BruteForce, SSH-Bruteforce      |
| 15-02 | DoS-GoldenEye, DoS-Slowloris        |
| 16-02 | DoS-SlowHTTPTest, DoS-Hulk          |
| 20-02 | DDoS-LOIC-HTTP, DDoS-LOIC-UDP       |
| 21-02 | DDoS-LOIC-UDP, DDoS-HOIC            |
| 22-02 | Web Brute Force, XSS, SQL Injection |
| 23-02 | Web Brute Force, XSS, SQL Injection |
| 28-02 | Infiltration                        |
| 01-03 | Infiltration                        |
| 02-03 | Bot                                 |

In [18]:
bruteforce = "Wednesday-14-02-2018"
dos_golden = "Thursday-15-02-2018"
dos_hulk = "Friday-16-02-2018"
ddos_http = "Tuesday-20-02-2018"
ddos_udp = "Wednesday-21-02-2018"
web_first = "Thursday-22-02-2018"
web_second = "Friday-23-02-2018"
infiltration_first = "Wednesday-28-02-2018"
infiltration_second = "Thursday-01-03-2018"
botnet = "Friday-02-03-2018"

In [14]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,bruteforce)

Found 11 files for Wednesday-14-02-2018
Processing [11/11] Wednesday-14-02-2018_TrafficForML_CICFlowMeter_00011.parquet...
Completed.


In [15]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,dos_golden)

Found 11 files for Thursday-15-02-2018
Processing [11/11] Thursday-15-02-2018_TrafficForML_CICFlowMeter_00011.parquet...
Completed.


In [16]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,dos_hulk)

Found 11 files for Friday-16-02-2018
Processing [11/11] Friday-16-02-2018_TrafficForML_CICFlowMeter_00011.parquet...
Completed.


In [19]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,ddos_http)

Found 80 files for Tuesday-20-02-2018
Processing [80/80] Tuesday-20-02-2018_TrafficForML_CICFlowMeter_00080.parquet...
Completed.


In [20]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,ddos_udp)

Found 11 files for Wednesday-21-02-2018
Processing [11/11] Wednesday-21-02-2018_TrafficForML_CICFlowMeter_00011.parquet...
Completed.


In [21]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,web_first)

Found 11 files for Thursday-22-02-2018
Processing [11/11] Thursday-22-02-2018_TrafficForML_CICFlowMeter_00011.parquet...
Completed.


In [22]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,web_second)

Found 11 files for Friday-23-02-2018
Processing [11/11] Friday-23-02-2018_TrafficForML_CICFlowMeter_00011.parquet...
Completed.


In [23]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,infiltration_first)

Found 7 files for Wednesday-28-02-2018
Processing [7/7] Wednesday-28-02-2018_TrafficForML_CICFlowMeter_00007.parquet...
Completed.


In [24]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,infiltration_second)

Found 4 files for Thursday-01-03-2018
Processing [4/4] Thursday-01-03-2018_TrafficForML_CICFlowMeter_00004.parquet...
Completed.


In [25]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,botnet)

Found 11 files for Friday-02-03-2018
Processing [11/11] Friday-02-03-2018_TrafficForML_CICFlowMeter_00011.parquet...
Completed.


Files are successfully split.

### Imputation

imputation for NaN